# Notebook T4.1 — Pipeline de Clasificación + Explicación con LLM
## Patrón: "El modelo predice, el LLM explica"

**Referencia en la guía metodológica:** PASO 5 → PATRÓN A  
**Nivel:** Principiante en GenAI (se asume experiencia en ML clásico)

---

### ¿Qué vas a aprender en este notebook?

Este notebook demuestra el patrón más habitual para **combinar ML clásico con IA Generativa**: 
usas un modelo de Machine Learning para hacer predicciones rápidas y escalables, 
y luego usas un LLM para traducir esas predicciones a lenguaje natural que cualquier persona 
(no técnica) pueda entender y actuar sobre ella.

**Caso de uso:** Predicción de churn (abandono) de clientes en una empresa.  
El modelo ML dice "este cliente va a abandonar". El LLM explica *por qué* y recomienda *qué hacer*.

---

### Flujo completo del pipeline

```
Datos del cliente (features)
         ↓
  RandomForestClassifier    ← entrenado con scikit-learn
         ↓
  predicción + probabilidad
         ↓
  Prompt con todo el contexto
         ↓
  LLM (Gemini/Claude)
         ↓
  Explicación accionable en español
```

---

### Conexión con la guía metodológica (001_Proceso_IA_Generativa_Claude)

- **Paso 1:** El problema es clasificación binaria (churn/no churn) + necesidad de explicabilidad  
- **Paso 2:** Configuramos LangChain + Gemini con temperatura baja (explicaciones consistentes)  
- **Paso 3:** El prompt tiene rol (analista), contexto (perfil del cliente) y formato de salida  
- **Paso 4:** Dataset sintético tabular — datos numéricos y categóricos  
- **Paso 5:** PATRÓN A — ML predice → LLM explica  
- **Paso 8:** Pipeline encapsulado en función `predecir_y_explicar()`

## PASO 2 — Instalación y Configuración

### ¿Por qué estas librerías?

- **langchain**: framework que conecta LLMs con tu código Python de forma estructurada
- **langchain-google-genai**: conector específico para usar Gemini de Google
- **scikit-learn**: el ML clásico que ya conoces (RandomForest, métricas, etc.)
- **python-dotenv**: para cargar API keys desde un archivo `.env` sin hardcodearlas en el código

In [ ]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

## Configuración de la API Key

### ¿Cómo funciona la autenticación con el LLM?

Cuando llamas a Gemini o Claude, estás haciendo una petición a un servidor remoto (API).  
Necesitas una **clave de autenticación** que identifique tu cuenta y permite facturar el uso.

**Opciones de carga:**
1. **Archivo `.env`** (recomendado en local): crea un archivo `.env` con `GOOGLE_API_KEY=tu_clave`
2. **Variables de entorno del sistema**: exportas la variable en tu terminal
3. **Google Colab**: usas `userdata.get()` para leer los secrets del notebook

> ⚠️ **Nunca** pongas la API key directamente en el código. Si subes el notebook a GitHub, 
> cualquiera podría usarla y acumular cargos en tu cuenta.

In [2]:
import os
from dotenv import load_dotenv

# Lee el archivo .env si existe
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

# En Google Colab usa: from google.colab import userdata; GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
# En local con .env usa: GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")
GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")
print("✅ API Key cargada")

✅ API Key cargada


## PASO 4 — Dataset Sintético de Churn

### ¿Por qué crear datos sintéticos?

En proyectos reales usarías datos de tu empresa. Aquí creamos datos sintéticos para:
- No depender de datos reales con información sensible
- Controlar exactamente las reglas que generan el target (churn)
- Hacer el ejemplo reproducible y comprensible

### ¿Qué es el churn?

**Churn** = cliente que abandona o cancela el servicio. Es una métrica crítica en 
negocios de suscripción (SaaS, telecomunicaciones, streaming).  
Predecirlo con antelación permite actuar antes de que el cliente se vaya.

### Features del dataset

| Feature | Tipo | Descripción |
|---------|------|-------------|
| `antiguedad_meses` | Numérico | Cuánto tiempo lleva el cliente |
| `num_productos` | Numérico | Cuántos productos tiene contratados |
| `saldo_promedio` | Continuo | Saldo medio de su cuenta |
| `llamadas_soporte` | Numérico | Llamadas al soporte el último mes (señal de frustración) |
| `uso_app_mensual` | Numérico | Días al mes que usa la app (señal de engagement) |
| `plan` | Categórico | Tipo de suscripción |
| `churn` | Binario | **Target**: 1=abandona, 0=se queda |

### La regla sintética de churn

El churn se define con una regla explícita que simula comportamientos reales:
- Muchas llamadas al soporte → frustración → churn
- Uso mínimo de la app → desenganche → churn
- Saldo muy bajo + cliente nuevo → poca vinculación → churn

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# Semilla para reproducibilidad — siempre fíjala al crear datos sintéticos
np.random.seed(42)
n = 500

df = pd.DataFrame({
    "antiguedad_meses": np.random.randint(1, 60, n),
    "num_productos": np.random.randint(1, 5, n),
    "saldo_promedio": np.random.uniform(0, 50000, n).round(2),
    "llamadas_soporte": np.random.randint(0, 10, n),
    "uso_app_mensual": np.random.randint(0, 30, n),
    "plan": np.random.choice(["básico", "estándar", "premium"], n),
})

# La regla de churn es una combinación de condiciones con OR lógico
# OR = basta con que se cumpla UNA de las condiciones para hacer churn
df["churn"] = (
    (df["llamadas_soporte"] > 6) |       # más de 6 llamadas → muy frustrado
    (df["uso_app_mensual"] < 3) |        # menos de 3 días → desenganchado
    ((df["saldo_promedio"] < 500) & (df["antiguedad_meses"] < 6))  # nuevo + poca vinculación
).astype(int)

print(f"Dataset creado: {df.shape[0]} clientes, {df.shape[1]} columnas")
print(f"Tasa de churn: {df['churn'].mean():.1%}  →  {df['churn'].sum()} clientes en riesgo")
df.head()

Dataset creado: 500 clientes, 7 columnas
Tasa de churn: 38.8%  →  194 clientes en riesgo


,antiguedad_meses,num_productos,saldo_promedio,llamadas_soporte,uso_app_mensual,plan,churn
0,39,2,27771.59,1,10,premium,0
1,52,4,38449.37,2,9,premium,0
2,29,3,47238.29,4,29,premium,0
3,15,3,42482.37,7,17,premium,1
4,43,2,12367.41,9,17,básico,1


## PASO 5 (parte A) — Entrenar el Clasificador ML

### ¿Por qué Random Forest para este problema?

El **RandomForestClassifier** es un buen punto de partida para clasificación tabular porque:
- Maneja bien variables categóricas (tras encoding) y numéricas mezcladas
- Es robusto al overfitting gracias al ensemble de muchos árboles
- Proporciona `predict_proba()` que nos da la **probabilidad** de churn, no solo 0/1
- La probabilidad es clave: pasaremos al LLM el porcentaje de riesgo, no solo la etiqueta

### LabelEncoder: convirtiendo texto a números para el ML

Los modelos ML no entienden texto. `LabelEncoder` convierte categorías a enteros:
- "básico" → 0
- "estándar" → 1  
- "premium" → 2

Guardamos el encoder (`le`) para poder usar `le.transform()` más tarde con clientes nuevos.

In [4]:
le = LabelEncoder()
df["plan_encoded"] = le.fit_transform(df["plan"])

features = ["antiguedad_meses", "num_productos", "saldo_promedio",
            "llamadas_soporte", "uso_app_mensual", "plan_encoded"]

X = df[features]
y = df["churn"]

# 80% entrenamiento, 20% test — estándar en ML supervisado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# n_estimators=100: el bosque tiene 100 árboles (más = más robusto pero más lento)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]  # columna 1 = probabilidad de clase "churn"

print("📊 Métricas del clasificador en el conjunto de test:")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
print("\n💡 Nota: Con datos sintéticos basados en reglas simples, el modelo tiende a 100%")
print("   En datos reales esperarías accuracy del 75-90% dependiendo del problema")

📊 Métricas del clasificador en el conjunto de test:
              precision    recall  f1-score   support

    No Churn       1.00      1.00      1.00        62
       Churn       1.00      1.00      1.00        38

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100


💡 Nota: Con datos sintéticos basados en reglas simples, el modelo tiende a 100%
   En datos reales esperarías accuracy del 75-90% dependiendo del problema


## PASO 5 (parte B) — Configurar LangChain + LLM y diseñar el Prompt

### LangChain Expression Language (LCEL): el operador `|`

LangChain usa el operador `|` (pipe) para encadenar pasos, igual que en Unix.  
La sintaxis `prompt | llm | parser` significa:
1. El prompt formatea el input como mensaje para el LLM
2. El LLM procesa el mensaje y devuelve una respuesta
3. El parser extrae el texto de la respuesta

Esto se llama **Chain** y es la unidad básica de composición en LangChain.

### Anatomía del prompt de explicación

El prompt tiene estas partes (ver guía metodológica PASO 3):

1. **Rol**: "Eres un analista de retención de clientes" → el LLM sabe desde qué perspectiva hablar
2. **Contexto**: el perfil del cliente con todas sus métricas
3. **Predicción del modelo**: qué decidió el ML y con qué confianza
4. **Formato de salida estructurado**: 3 secciones específicas que facilitan la lectura

### ¿Por qué temperatura 0.3?

- Queremos explicaciones **coherentes y profesionales**, no creativas
- Temperatura baja (0-0.3) → el LLM elige siempre las palabras más probables → más consistente
- Si pusieramos 0.8, cada ejecución daría una explicación diferente (útil para brainstorming, no aquí)

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Inicializar el LLM (ver guía metodológica PASO 2)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3  # baja para explicaciones consistentes
)

# Prompt de explicación (ver guía metodológica PASO 3)
# Las variables entre {} se rellenan en cada llamada con chain.invoke({...})
# Las llaves dobles {{ }} son literales en el template (no variables)
prompt = ChatPromptTemplate.from_template("""
Eres un analista de retención de clientes. Dado el perfil de un cliente y la predicción del modelo,
genera una explicación clara y accionable en español.

**Perfil del cliente:**
- Antigüedad: {antiguedad_meses} meses
- Número de productos contratados: {num_productos}
- Saldo promedio: ${saldo_promedio:,.0f}
- Llamadas al soporte (último mes): {llamadas_soporte}
- Uso de la app (días/mes): {uso_app_mensual}
- Plan: {plan}

**Predicción del modelo:** {prediccion} (probabilidad de churn: {probabilidad:.0%})

Responde con:
1. **Diagnóstico** (2 oraciones): Por qué el modelo tomó esta decisión
2. **Señales de alerta**: Lista los factores de riesgo principales
3. **Acción recomendada**: Qué debería hacer el equipo de retención
""")

# La chain encadena: prompt → llm → parser de texto
chain = prompt | llm | StrOutputParser()

print("✅ Chain de explicación configurada correctamente")

c:\Users\Oscar\OneDrive - FM4\Escritorio\EVOLVE\Data Science\.venv_tf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Chain de explicación configurada correctamente


## PASO 5 (parte C) — Ejecutar el Pipeline: Predicción + Explicación

### ¿Qué hace este bloque de código?

1. Toma el conjunto de test y añade las predicciones del modelo ML
2. Filtra los clientes que el modelo predice como churn
3. Para los primeros 3 casos, llama al LLM para generar una explicación
4. Imprime la explicación formateada

### El flujo de datos concreto

```
X_test (features numéricas)
    → clf.predict()    → y_pred (0 o 1)
    → clf.predict_proba()[:, 1]  → y_prob (probabilidad entre 0 y 1)
    → chain.invoke({features + prediccion + probabilidad})
    → explicacion en texto
```

In [6]:
# Preparar el dataframe de test con predicciones y metadatos
X_test_df = X_test.copy()
X_test_df["prediccion_num"] = y_pred
X_test_df["probabilidad"] = y_prob
# Recuperar el plan original (texto, no codificado) usando los índices originales
X_test_df["plan"] = df.loc[X_test.index, "plan"].values

# Filtrar solo los clientes predichos como churn y tomar los 3 primeros
casos_churn = X_test_df[X_test_df["prediccion_num"] == 1].head(3)

print(f"Clientes predichos como CHURN en test: {(y_pred == 1).sum()}")
print(f"Generando explicaciones para los 3 casos de mayor riesgo...\n")

for i, (idx, row) in enumerate(casos_churn.iterrows()):
    print(f"\n{'='*60}")
    print(f"📋 CLIENTE #{i+1} (ID interno: {idx})")
    print(f"{'='*60}")

    # Aquí ocurre la llamada al LLM — puede tardar 1-3 segundos por cliente
    explicacion = chain.invoke({
        "antiguedad_meses": int(row["antiguedad_meses"]),
        "num_productos": int(row["num_productos"]),
        "saldo_promedio": float(row["saldo_promedio"]),
        "llamadas_soporte": int(row["llamadas_soporte"]),
        "uso_app_mensual": int(row["uso_app_mensual"]),
        "plan": row["plan"],
        "prediccion": "CHURN" if row["prediccion_num"] == 1 else "NO CHURN",
        "probabilidad": float(row["probabilidad"])
    })

    print(explicacion)

Clientes predichos como CHURN en test: 38
Generando explicaciones para los 3 casos de mayor riesgo...


📋 CLIENTE #1 (ID interno: 73)
Aquí tienes el análisis de retención para el perfil del cliente proporcionado:

---

1.  **Diagnóstico**
    El modelo predice una probabilidad de churn extremadamente alta (97%) impulsada por un nivel alarmante de frustración y problemas no resueltos, reflejado en las 8 llamadas al soporte durante el último mes. A pesar de su alta interacción y valor, este cliente está experimentando una fricción significativa que lo empuja hacia la salida.

2.  **Señales de alerta**
    *   **8 llamadas al soporte en el último mes:** Este es el factor más crítico, indicando una frustración severa o problemas persistentes que no han sido resueltos satisfactoriamente. Es un claro indicador de una experiencia negativa.
    *   **Contradicción entre alta interacción y frustración:** El cliente tiene 4 productos, un saldo considerable y usa la app 20 días al mes. Esto sugie

## PASO 8 — Pipeline Completo Encapsulado como Función Reutilizable

### ¿Por qué encapsular en una función?

Al encapsular todo el pipeline en `predecir_y_explicar()`, cualquier parte de tu sistema 
puede llamarla con solo pasar un diccionario con los datos del cliente.  

Esto es el PASO 8 de la guía metodológica: **código limpio, reutilizable y con manejo de errores**.

La función hace dos cosas en secuencia:
1. **Predicción ML**: transforma los datos del cliente al formato que espera el modelo entrenado
2. **Explicación LLM**: pasa los datos + la predicción al LLM para generar el diagnóstico

### ¿Por qué `le.transform()` en lugar de `le.fit_transform()`?

- `fit_transform()` = aprende el encoding Y lo aplica (solo en entrenamiento)
- `transform()` = aplica el encoding ya aprendido (en producción con datos nuevos)
- Si usáramos `fit_transform()` con un cliente nuevo, el encoder podría asignar números diferentes

In [7]:
def predecir_y_explicar(cliente: dict) -> dict:
    """
    Pipeline completo: recibe datos de un cliente, predice churn y genera explicación.
    
    Args:
        cliente: diccionario con keys: antiguedad_meses, num_productos, saldo_promedio,
                 llamadas_soporte, uso_app_mensual, plan
    
    Returns:
        diccionario con: prediccion (str), probabilidad_churn (str), explicacion (str)
    """
    # PASO A: Transformar la feature categórica usando el encoder ya entrenado
    plan_enc = le.transform([cliente["plan"]])[0]
    
    # PASO B: Construir el vector de features en el orden exacto que espera el modelo
    features_input = [[
        cliente["antiguedad_meses"],
        cliente["num_productos"],
        cliente["saldo_promedio"],
        cliente["llamadas_soporte"],
        cliente["uso_app_mensual"],
        plan_enc
    ]]

    # PASO C: Predicción del modelo ML
    pred = clf.predict(features_input)[0]
    prob = clf.predict_proba(features_input)[0][1]

    # PASO D: Explicación del LLM — pasamos tanto los datos como la predicción
    explicacion = chain.invoke({
        **cliente,  # desempaqueta el diccionario como variables del prompt
        "prediccion": "CHURN" if pred == 1 else "NO CHURN",
        "probabilidad": prob
    })

    return {
        "prediccion": "CHURN" if pred == 1 else "NO CHURN",
        "probabilidad_churn": f"{prob:.1%}",
        "explicacion": explicacion
    }


# --- Prueba con un cliente nuevo (no estaba en el dataset de entrenamiento) ---
nuevo_cliente = {
    "antiguedad_meses": 3,       # cliente muy nuevo
    "num_productos": 1,          # poca vinculación
    "saldo_promedio": 200,       # saldo muy bajo
    "llamadas_soporte": 8,       # muy frustrado
    "uso_app_mensual": 1,        # casi no usa la app
    "plan": "básico"
}

print("🔍 Analizando cliente nuevo...")
resultado = predecir_y_explicar(nuevo_cliente)
print(f"\n{'='*60}")
print(f"Predicción: {resultado['prediccion']}")
print(f"Probabilidad de churn: {resultado['probabilidad_churn']}")
print(f"{'='*60}")
print(f"\n{resultado['explicacion']}")

🔍 Analizando cliente nuevo...


c:\Users\Oscar\OneDrive - FM4\Escritorio\EVOLVE\Data Science\.venv_tf\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Oscar\OneDrive - FM4\Escritorio\EVOLVE\Data Science\.venv_tf\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



Predicción: CHURN
Probabilidad de churn: 94.0%

Aquí tienes el análisis para el perfil del cliente presentado:

---

1.  **Diagnóstico:**
    El modelo predice un churn con una probabilidad extremadamente alta debido a una combinación crítica de alta frustración y baja interacción con los servicios digitales. La elevada frecuencia de llamadas al soporte, junto con un uso casi nulo de la aplicación, sugiere que el cliente no está encontrando valor o solución a sus problemas, lo que es especialmente preocupante en un cliente de reciente antigüedad.

2.  **Señales de alerta:**
    *   **Alta frecuencia de llamadas al soporte (8 en el último mes):** Indica problemas recurrentes, frustración significativa o incapacidad para resolver sus inquietudes por otros medios.
    *   **Mínimo uso de la aplicación (1 día/mes):** Sugiere que el cliente no está aprovechando los canales digitales, no encuentra valor en ellos o no sabe cómo utilizarlos, perdiendo una vía clave de auto-servicio y engageme

## Conclusiones del Notebook T4.1

### Lo que has aprendido

1. **Patrón A de la guía metodológica** funciona en 4 pasos:
   - Entrena un modelo ML con scikit-learn (lo que ya sabes)
   - El modelo genera predicción + probabilidad
   - Construyes un prompt con los datos + la predicción
   - El LLM genera la explicación en lenguaje natural

2. **LangChain LCEL** (`prompt | llm | parser`) es la forma estándar de encadenar pasos en GenAI

3. **La temperatura** controla la creatividad: 0.3 para explicaciones profesionales y consistentes

4. **Encapsulamiento** en funciones (PASO 8 de la guía): el pipeline se convierte en una función
   que cualquier sistema puede llamar con `predecir_y_explicar(datos_cliente)`

---

### ¿Cuándo usar este patrón en proyectos reales?

- Sistemas de crédito: el modelo aprueba/deniega, el LLM explica al cliente por qué
- Detección de fraude: el modelo alerta, el LLM explica al analista qué revisar
- Diagnóstico médico asistido: el modelo predice, el LLM resume los factores de riesgo
- Cualquier caso donde usuarios no técnicos necesiten entender predicciones ML

---

### Patrón clave (para memorizar)

```
ML Model → predicción + features → Prompt → LLM → Explicación accionable
```